# EEG Seizure Analysis

This notebook examined selected scalp EEG recordings from the CHB-MIT dataset. I used eight recordings from one subject to explore signal patterns across annotated seizure and non-seizure periods.

The analysis focused on signal amplitude, variation, and frequency-band power. I calculated a set of features for each EEG channel and non-overlapping 2-second window and exported the results for exploration in Tableau.

**Dataset:** CHB-MIT Scalp EEG Database, available through PhysioNet  
**Recordings used:**
- `chb01_01.edf`
- `chb01_03.edf`
- `chb01_04.edf`
- `chb01_15.edf`
- `chb01_16.edf`
- `chb01_18.edf`
- `chb01_21.edf`
- `chb01_26.edf`

The CHB01 subject summary provided the seizure timing. The recordings contained 23 EEG channels and used a sampling rate of 256 Hz.

The analysis used eight recordings from one subject, so the channel-window observations should not be interpreted as independent subjects or seizure events.

## 1. Environment and reproducibility

I used Python and the MNE, NumPy, Pandas, and SciPy libraries. The notebook processed the eight selected EDF recordings and the CHB01 summary file.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
from scipy.signal import butter, sosfiltfilt, welch

# Google Colab upload helper (skipped automatically outside Colab)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RECORDINGS = [
    "chb01_01.edf",
    "chb01_03.edf",
    "chb01_04.edf",
    "chb01_15.edf",
    "chb01_16.edf",
    "chb01_18.edf",
    "chb01_21.edf",
    "chb01_26.edf",
]
SUMMARY_FILE = "chb01-summary.txt"
OUTPUT_DIR = Path("eeg_tableau_exports")
OUTPUT_DIR.mkdir(exist_ok=True)

if IN_COLAB:
    required = RECORDINGS + [SUMMARY_FILE]
    missing = [name for name in required if not Path(name).exists()]
    if missing:
        print("Upload the eight EDF files and chb01-summary.txt.")
        files.upload()

missing = [name for name in RECORDINGS + [SUMMARY_FILE] if not Path(name).exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files: " + ", ".join(missing) +
        ". Place all eight EDF files and chb01-summary.txt in the working directory."
    )

print(f"Found {len(RECORDINGS)} EDF recordings and the annotation summary.")

## 2. Read the subject summary and check seizure annotations

The CHB01 summary file listed seizure start and end times relative to the beginning of each EDF recording. I read those values from the summary file and used them to label the 2-second windows during feature extraction.

In [ ]:
def parse_chb_summary(summary_path, selected_recordings):
    text = Path(summary_path).read_text(errors="replace")
    events = []

    # Split the summary into file-specific sections.
    sections = re.split(r"(?=File Name:\s*chb\d+_\d+\.edf)", text)
    for section in sections:
        name_match = re.search(r"File Name:\s*(chb\d+_\d+\.edf)", section)
        if not name_match:
            continue
        recording = name_match.group(1)
        if recording not in selected_recordings:
            continue

        starts = re.findall(r"Seizure Start Time:\s*(\d+)\s*seconds", section)
        ends = re.findall(r"Seizure End Time:\s*(\d+)\s*seconds", section)

        if len(starts) != len(ends):
            raise ValueError(f"Unmatched seizure start/end annotations for {recording}")

        for number, (start, end) in enumerate(zip(starts, ends), start=1):
            events.append({
                "recording": recording,
                "seizure_number": number,
                "seizure_start_sec": int(start),
                "seizure_end_sec": int(end),
                "duration_sec": int(end) - int(start),
                "subject": "chb01",
                "annotation_source": "CHB-MIT chb01-summary.txt",
            })

    return pd.DataFrame(events, columns=[
        "recording", "seizure_number", "seizure_start_sec",
        "seizure_end_sec", "duration_sec", "subject", "annotation_source"
    ])

annotations = parse_chb_summary(SUMMARY_FILE, RECORDINGS)
display(annotations)
assert len(annotations) == 7, f"Expected 7 annotated events, found {len(annotations)}" 

## 3. Preprocessing and feature extraction

I applied a 1–40 Hz Butterworth band-pass filter to each EEG channel. I divided the recordings into non-overlapping 2-second windows and calculated the following features for each channel and window:

- mean and standard deviation of amplitude;
- root mean square (RMS);
- line length, which provided a simple measure of signal variation;
- relative power in the delta, theta, alpha, beta, and 30–40 Hz frequency bands;
- spectral entropy.

I labelled a window as seizure (1) when its midpoint fell within an annotated seizure interval. The start of the interval was included and the end was excluded.

The gamma feature covered 30–40 Hz because the signal was filtered at 40 Hz.

In [ ]:
WINDOW_SEC = 2
LOW_CUT_HZ = 1.0
HIGH_CUT_HZ = 40.0
FILTER_ORDER = 4

BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma_30_40": (30, 40),
}

def butterworth_bandpass(data, sfreq, low=LOW_CUT_HZ, high=HIGH_CUT_HZ, order=FILTER_ORDER):
    nyquist = sfreq / 2.0
    sos = butter(order, [low / nyquist, high / nyquist],
                 btype="bandpass", output="sos")
    return sosfiltfilt(sos, data, axis=-1)

def calculate_features(x, sfreq):
    # Welch PSD for the 2-second channel segment.
    freqs, psd = welch(x, fs=sfreq, nperseg=min(len(x), int(sfreq)), detrend="constant")
    total_power = np.trapz(psd, freqs)

    features = {
        "mean_uv": float(np.mean(x)),
        "std_uv": float(np.std(x)),
        "rms_uv": float(np.sqrt(np.mean(x ** 2))),
        "line_length": float(np.mean(np.abs(np.diff(x)))),
    }

    for band_name, (low, high) in BANDS.items():
        mask = (freqs >= low) & (freqs < high)
        band_power = np.trapz(psd[mask], freqs[mask]) if mask.sum() > 1 else 0.0
        features[f"rel_power_{band_name}"] = (
            float(band_power / total_power) if total_power > 0 else np.nan
        )

    psd_sum = np.sum(psd)
    probabilities = psd / psd_sum if psd_sum > 0 else np.zeros_like(psd)
    probabilities = probabilities[probabilities > 0]
    features["spectral_entropy"] = (
        float(-np.sum(probabilities * np.log2(probabilities)))
        if len(probabilities) else np.nan
    )
    return features

def window_midpoint_is_seizure(start_sec, end_sec, seizure_intervals):
    midpoint = (start_sec + end_sec) / 2.0
    return int(any(
        event_start <= midpoint < event_end
        for event_start, event_end in seizure_intervals
    ))

def extract_recording_features(recording, raw, seizure_intervals):
    sfreq = float(raw.info["sfreq"])
    if not np.isclose(sfreq, 256.0):
        raise ValueError(f"{recording}: expected 256 Hz, found {sfreq}")

    # Use the 23 EEG channels and convert MNE's volts to microvolts.
    eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])
    if len(eeg_picks) != 23:
        raise ValueError(f"{recording}: expected 23 EEG channels, found {len(eeg_picks)}")

    data = raw.get_data(picks=eeg_picks) * 1e6
    channel_names = [raw.ch_names[i] for i in eeg_picks]
    filtered = butterworth_bandpass(data, sfreq)

    samples_per_window = int(WINDOW_SEC * sfreq)
    n_windows = filtered.shape[1] // samples_per_window
    rows = []

    for window_index in range(n_windows):
        start_sample = window_index * samples_per_window
        end_sample = start_sample + samples_per_window
        start_sec = start_sample / sfreq
        end_sec = end_sample / sfreq
        label = window_midpoint_is_seizure(start_sec, end_sec, seizure_intervals)

        for channel_index, channel_name in enumerate(channel_names):
            x = filtered[channel_index, start_sample:end_sample]
            row = {
                "recording": recording,
                "subject": "chb01",
                "channel": channel_name,
                "window_start_sec": int(start_sec),
                "window_end_sec": int(end_sec),
                "window_mid_sec": (start_sec + end_sec) / 2,
                "seizure_label": label,
                "seizure_status": "Seizure" if label else "Non-seizure",
            }
            row.update(calculate_features(x, sfreq))
            rows.append(row)

    return pd.DataFrame(rows), {
        "recording": recording,
        "sampling_rate_hz": sfreq,
        "channels": len(channel_names),
        "duration_sec": int(raw.n_times / sfreq),
        "windows_2sec": n_windows,
        "annotated_seizures": len(seizure_intervals),
    }

## 4. Process the selected recordings

I processed the full duration of each of the eight selected EDF recordings. For each recording, I calculated the features for every EEG channel and non-overlapping 2-second window. The resulting table contained one row for each channel-window observation.

In [ ]:
event_lookup = {
    recording: list(zip(
        group["seizure_start_sec"].astype(float),
        group["seizure_end_sec"].astype(float)
    ))
    for recording, group in annotations.groupby("recording")
}
for recording in RECORDINGS:
    event_lookup.setdefault(recording, [])

feature_tables = []
coverage_rows = []

for recording in RECORDINGS:
    print(f"Processing {recording} ...")
    raw = mne.io.read_raw_edf(recording, preload=True, verbose="ERROR")
    table, coverage = extract_recording_features(
        recording, raw, event_lookup[recording]
    )
    feature_tables.append(table)
    coverage_rows.append(coverage)
    del raw

features = pd.concat(feature_tables, ignore_index=True)
coverage = pd.DataFrame(coverage_rows)

print("Feature table:", features.shape)
print("Unique time windows:", features[["recording", "window_start_sec"]].drop_duplicates().shape[0])
display(coverage)
display(features.head())

## 5. Check the extracted data

I checked the extracted table against the expected number of recordings, EEG channels, time windows, and seizure-labelled channel-window observations. I also checked the recording coverage and the number of annotated seizure events.

In [ ]:
expected_windows_by_recording = {
    "chb01_01.edf": 1800,
    "chb01_03.edf": 1800,
    "chb01_04.edf": 1800,
    "chb01_15.edf": 1800,
    "chb01_16.edf": 1800,
    "chb01_18.edf": 1800,
    "chb01_21.edf": 1800,
    "chb01_26.edf": 1162,
}

assert features["recording"].nunique() == 8
assert features["channel"].nunique() == 23
assert len(annotations) == 7
assert coverage.set_index("recording")["windows_2sec"].to_dict() == expected_windows_by_recording
assert len(features) == 316_526
assert features[["recording", "window_start_sec"]].drop_duplicates().shape[0] == 13_762
assert int(features["seizure_label"].sum()) == 5_106

print("Validation passed.")
print(f"Rows: {len(features):,}")
print(f"Unique recording-time windows: {features[['recording', 'window_start_sec']].drop_duplicates().shape[0]:,}")
print(f"Seizure-labeled channel-window rows: {int(features['seizure_label'].sum()):,}")
print(f"Annotated seizure events: {len(annotations)}")

## 6. Summarize the features by EEG channel

I summarized the extracted features by seizure status and EEG channel. These values provided a channel-level view of the differences between seizure and non-seizure observations for exploration in Tableau.

In [ ]:
feature_columns = [
    "rms_uv", "std_uv", "line_length",
    "rel_power_delta", "rel_power_theta", "rel_power_alpha",
    "rel_power_beta", "rel_power_gamma_30_40", "spectral_entropy",
]

channel_summary = (
    features.groupby(["seizure_status", "channel"], as_index=False)
    .agg(
        window_count=("seizure_label", "size"),
        mean_rms_uv=("rms_uv", "mean"),
        median_rms_uv=("rms_uv", "median"),
        mean_std_uv=("std_uv", "mean"),
        mean_line_length=("line_length", "mean"),
        mean_rel_power_delta=("rel_power_delta", "mean"),
        mean_rel_power_theta=("rel_power_theta", "mean"),
        mean_rel_power_alpha=("rel_power_alpha", "mean"),
        mean_rel_power_beta=("rel_power_beta", "mean"),
        mean_rel_power_gamma_30_40=("rel_power_gamma_30_40", "mean"),
        mean_spectral_entropy=("spectral_entropy", "mean"),
    )
)
display(channel_summary.head())

## 7. Model evaluation

I did not estimate predictive performance or feature importance. The analysis used recordings from one subject, and channel-window observations from the same recordings were related. A random window-level split could place related observations in both the training and test sets, so it would not provide a reliable estimate of performance on unseen subjects.

In [ ]:
model_metrics = pd.DataFrame([{
    "status": "Not estimated",
    "reason": (
        "All eight recordings are from one subject. Random window-level splitting "
        "risks temporal/channel leakage and cannot estimate generalization to unseen subjects."
    ),
    "recordings": len(RECORDINGS),
    "unique_seizure_events": len(annotations),
    "subjects": 1,
    "window_seconds": WINDOW_SEC,
    "overlap": np.nan,
    "recommended_next_step": (
        "Add multiple subjects and evaluate using subject-wise splits."
    ),
}])
display(model_metrics)

## 8. Export data for Tableau

I exported the extracted window-level features, seizure annotations, recording coverage, and channel-level summaries for exploration in Tableau.

The window-level table contained one row for each EEG channel and 2-second window. The event table contained the annotated seizure intervals. The remaining tables summarized recording coverage, channel-level features, and the model evaluation note.

The exported files were:
- `eeg_channel_window_features_tableau.csv`
- `seizure_events_tableau.csv`
- `recording_coverage.csv`
- `channel_feature_summary_tableau.csv`
- `model_metrics_exploratory.csv`

In [ ]:
exports = {
    "eeg_channel_window_features_tableau.csv": features,
    "seizure_events_tableau.csv": annotations,
    "recording_coverage.csv": coverage,
    "channel_feature_summary_tableau.csv": channel_summary,
    "model_metrics_exploratory.csv": model_metrics,
}

for filename, dataframe in exports.items():
    output_path = OUTPUT_DIR / filename
    dataframe.to_csv(output_path, index=False)
    print(f"{filename}: {len(dataframe):,} rows ({output_path.stat().st_size / 1024:.1f} KB)")

print("\nExport folder:", OUTPUT_DIR.resolve())

## 9. Notes on scope

This analysis used the full duration of eight selected CHB01 recordings, not the full CHB-MIT dataset. All recordings belonged to one subject. The channel-window observations were repeated measurements within the selected recordings and did not represent independent subjects or seizure events.

The extracted features described selected signal characteristics in the recordings. The analysis was exploratory and did not establish whether the same patterns would be present in other subjects or recording settings.